# 01_generate_dataset.ipynb
Genera TODOS los datasets necesarios para el proyecto:
1. interactions.parquet - Interacciones usuario-juego
2. user2idx.json - Mapeo usuario -> índice
3. item2idx.json - Mapeo juego -> índice
4. (Los embeddings se generan en notebooks posteriores: 02, 07, 08)

In [1]:
import json
import pandas as pd
import numpy as np
import os
import ast  # Para parsear diccionarios de Python

PATH_JSON = "../Data/australian_user_reviews.json"
SAVE_INTERACTIONS = "../Data/interactions.parquet"
SAVE_USERMAP = "../Data/user2idx.json"
SAVE_ITEMMAP = "../Data/item2idx.json"

# 1) Parse JSON (en realidad son diccionarios Python línea por línea)
def parse_reviews(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Convertir string de dict Python a dict real
            entry = ast.literal_eval(line)
            user = entry.get("user_id")
            for r in entry.get("reviews", []):
                rows.append({
                    "user_id": user,
                    "item_id": r.get("item_id"),
                })
    return pd.DataFrame(rows)

df = parse_reviews(PATH_JSON)
print("Shape raw:", df.shape)
df.head()

Shape raw: (59305, 2)


,user_id,item_id
0,76561197970982479,1250
1,76561197970982479,22200
2,76561197970982479,43110
3,js41637,251610
4,js41637,227300


In [2]:
# 2) Crear índices
user2idx = {u:i for i,u in enumerate(df["user_id"].unique())}
item2idx = {g:i for i,g in enumerate(df["item_id"].unique())}

df["user_idx"] = df["user_id"].map(user2idx).astype("int32")
df["item_idx"] = df["item_id"].map(item2idx).astype("int32")

print(df.head())

             user_id item_id  user_idx  item_idx
0  76561197970982479    1250         0         0
1  76561197970982479   22200         0         1
2  76561197970982479   43110         0         2
3            js41637  251610         1         3
4            js41637  227300         1         4


In [3]:
# 3) Guardar archivos básicos
df.to_parquet(SAVE_INTERACTIONS, index=False)

with open(SAVE_USERMAP, "w") as f:
    json.dump({str(k):int(v) for k,v in user2idx.items()}, f)

with open(SAVE_ITEMMAP, "w") as f:
    json.dump({str(k):int(v) for k,v in item2idx.items()}, f)

print("✅ Archivos básicos guardados:")
print(f"   - {SAVE_INTERACTIONS}")
print(f"   - {SAVE_USERMAP}")
print(f"   - {SAVE_ITEMMAP}")

✅ Archivos básicos guardados:
   - ../Data/interactions.parquet
   - ../Data/user2idx.json
   - ../Data/item2idx.json


## Resumen de Datasets del Proyecto

Este notebook genera los **datasets base**. Otros datasets se generan en notebooks posteriores:

### 📦 Datasets Base (este notebook):
- ✅ `interactions.parquet` - Interacciones usuario-juego (user_id, item_id, user_idx, item_idx)
- ✅ `user2idx.json` - Mapeo user_id → índice numérico
- ✅ `item2idx.json` - Mapeo item_id → índice numérico

### 🧠 Embeddings (generados en otros notebooks):
- **Notebook 02 (`02_model_keras_rs.ipynb`)**:
  - `item_embeddings_rs.npy` - Embeddings de juegos del RS (64-dim)
  - `user_embeddings_rs.npy` - Embeddings de usuarios del RS (64-dim)

- **Notebook 07 (`07_regressor_review_embeddings.ipynb`)**:
  - `review_text_embeddings.npy` - Embeddings de texto de reviews (384-dim)
  - `game_texts_map.csv` - Mapeo item_idx → texto concatenado de reviews

- **Notebook 08 (`08_regressor_tag_embeddings.ipynb`)**:
  - `tag_embeddings.npy` - Embeddings de tags de juegos (384-dim)
  - `game_tags_map.csv` - Mapeo item_idx → tags concatenados

### 📄 Archivos de Entrada (NO se generan):
- `australian_user_reviews.json` - Reviews originales de usuarios (fuente externa)
- `steam_games.json` - Metadata de juegos de Steam (fuente externa)

In [4]:
import os

print("="*80)
print("VERIFICACIÓN DE DATASETS")
print("="*80)

# Verificar archivos generados en este notebook
files_to_check = [
    ("../Data/interactions.parquet", "Interacciones usuario-juego"),
    ("../Data/user2idx.json", "Mapeo usuarios"),
    ("../Data/item2idx.json", "Mapeo juegos"),
]

print("\n📦 Datasets Base (generados en ESTE notebook):")
for filepath, description in files_to_check:
    exists = os.path.exists(filepath)
    status = "✅" if exists else "❌"
    size = f"{os.path.getsize(filepath) / 1024:.1f} KB" if exists else "N/A"
    print(f"{status} {filepath:40s} - {description:30s} [{size}]")

# Verificar archivos que DEBERÍAN existir (generados en otros notebooks)
optional_files = [
    ("../Data/item_embeddings_rs.npy", "Item embeddings RS (Notebook 02)", "🧠"),
    ("../Data/user_embeddings_rs.npy", "User embeddings RS (Notebook 02)", "🧠"),
    ("../Data/review_text_embeddings.npy", "Review text embeddings (Notebook 07)", "🧠"),
    ("../Data/game_texts_map.csv", "Game texts mapping (Notebook 07)", "📄"),
    ("../Data/tag_embeddings.npy", "Tag embeddings (Notebook 08)", "🧠"),
    ("../Data/game_tags_map.csv", "Game tags mapping (Notebook 08)", "📄"),
]

print("\n🧠 Embeddings y Archivos Derivados (generados en OTROS notebooks):")
for filepath, description, icon in optional_files:
    exists = os.path.exists(filepath)
    status = "✅" if exists else "⚠️ "
    size = f"{os.path.getsize(filepath) / 1024:.1f} KB" if exists else "Pendiente"
    print(f"{status} {filepath:40s} - {description:40s} [{size}]")

# Verificar archivos de entrada
input_files = [
    ("../Data/australian_user_reviews.json", "Reviews originales"),
    ("../Data/steam_games.json", "Metadata de juegos"),
]

print("\n📥 Archivos de Entrada (deben existir ANTES de ejecutar):")
for filepath, description in input_files:
    exists = os.path.exists(filepath)
    status = "✅" if exists else "❌ FALTA"
    size = f"{os.path.getsize(filepath) / (1024*1024):.1f} MB" if exists else "N/A"
    print(f"{status} {filepath:40s} - {description:30s} [{size}]")

print("\n" + "="*80)
print("ESTADÍSTICAS:")
print("="*80)
print(f"Usuarios únicos: {len(user2idx):,}")
print(f"Juegos únicos: {len(item2idx):,}")
print(f"Interacciones totales: {len(df):,}")
print(f"Densidad: {len(df) / (len(user2idx) * len(item2idx)) * 100:.4f}%")
print("="*80)

VERIFICACIÓN DE DATASETS

📦 Datasets Base (generados en ESTE notebook):
✅ ../Data/interactions.parquet             - Interacciones usuario-juego    [796.5 KB]
✅ ../Data/user2idx.json                    - Mapeo usuarios                 [597.2 KB]
✅ ../Data/item2idx.json                    - Mapeo juegos                   [55.6 KB]

🧠 Embeddings y Archivos Derivados (generados en OTROS notebooks):
✅ ../Data/item_embeddings_rs.npy           - Item embeddings RS (Notebook 02)         [920.6 KB]
✅ ../Data/user_embeddings_rs.npy           - User embeddings RS (Notebook 02)         [6364.6 KB]
✅ ../Data/review_text_embeddings.npy       - Review text embeddings (Notebook 07)     [5523.1 KB]
✅ ../Data/game_texts_map.csv               - Game texts mapping (Notebook 07)         [13058.7 KB]
✅ ../Data/tag_embeddings.npy               - Tag embeddings (Notebook 08)             [4789.6 KB]
✅ ../Data/game_tags_map.csv                - Game tags mapping (Notebook 08)          [395.7 KB]

📥 Archivos de

## ¿Qué son los "reviews" en este contexto?

**IMPORTANTE:** En este proyecto, "review" = "interacción usuario-juego"

### 📊 Definición del Target:
```python
target_df = df.groupby("item_idx").size().reset_index(name="total_reviews")
```

**`total_reviews`** = **Número de usuarios que interactuaron con cada juego**

### 🎯 NO es:
- ❌ Texto de reviews
- ❌ Rating/puntuación
- ❌ Sentimiento de reviews

### ✅ ES:
- **Conteo de interacciones**: Cuántos usuarios dejaron algún tipo de review para ese juego
- **Proxy de popularidad**: Más interacciones = juego más popular
- **Variable continua**: Se predice con regresión (XGBoost)

### 📝 Ejemplo:
```
Juego A: 100 usuarios dejaron reviews → total_reviews = 100
Juego B: 5 usuarios dejaron reviews → total_reviews = 5
```

### 🔗 Conexión con otros notebooks:
- **Notebook 03-09**: Predicen `total_reviews` usando embeddings/metadata
- **Notebook 07**: Usa el TEXTO de los reviews para generar embeddings (diferente uso)
- **Notebook 08**: Usa los TAGS de los juegos para generar embeddings

## Verificar Datasets Generados